In [1]:
# Artificial and Computational Intelligence Assignment 1
## Problem solving by Uninformed & Informed Search

# List all the team members BITS ID ,Name along with % of contribution in this assignment: sample Provided below:
# 1. ___________________
# 2. __________________
# 3. ____________________
# 4. ___________________
# 5. ___________________

# 
# Things to follow
# 1.	Use appropriate data structures to represent the graph and the path using python libraries
# 2.	Provide proper documentation
# 3.	Find the path and print it

# Coding begins here

### 1.	Define the environment in the following block

# List the PEAS decription of the problem here in this markdown block
"""
P (Performance Measure): Minimum total cost/distance of the tour (shortest route).
E (Environment): Static, Discrete, Deterministic, Known (The map of Delhi with fixed costs).
A (Actuators): Movement along roads (Selecting the next destination).
S (Sensors): Current location, Road segment cost, Set of visited locations.
"""

# Design the agent as PSA Agent(Problem Solving Agent)
# Clear Initial data structures to define the graph and variable declarations is expected
# IMPORTATANT: Write distinct code block as below

import time
import sys
from collections import deque

# Setting recursion limit higher for DFS and RBFS
sys.setrecursionlimit(2000)

# Global variables for complexity analysis
DFS_NODES_EXPLORED = 0
RBFS_NODES_EXPLORED = 0
DFS_MAX_MEMORY = 0
RBFS_MAX_MEMORY = 0

# The Graph as an Adjacency List (Dictionary of Dictionaries)
# Cost is the edge weight
GRAPH = {
    "Indira Gandhi Airport": {"India Gate": 15, "Qutub Minar": 20},
    "India Gate": {"Indira Gandhi Airport": 15, "Red Fort": 10, "Lotus Temple": 15, "Humayun's Tomb": 10},
    "Qutub Minar": {"Indira Gandhi Airport": 20, "Humayun's Tomb": 20, "Lotus Temple": 15},
    "Red Fort": {"India Gate": 10, "Lotus Temple": 30, "Jama Masjid": 5},
    "Humayun's Tomb": {"Qutub Minar": 20, "India Gate": 10, "Lotus Temple": 10, "Akshardham Temple": 15},
    "Lotus Temple": {"India Gate": 15, "Qutub Minar": 15, "Red Fort": 30, "Humayun's Tomb": 10, "Akshardham Temple": 25},
    "Akshardham Temple": {"Humayun's Tomb": 15, "Lotus Temple": 25, "Jama Masjid": 20},
    "Jama Masjid": {"Red Fort": 5, "Akshardham Temple": 20}
}

ALL_LOCATIONS = set(GRAPH.keys())
START_NODE = "Indira Gandhi Airport"
NUM_LOCATIONS = len(ALL_LOCATIONS)

# Utility to get the size of data structures in bytes for memory tracking
def get_memory_size(obj):
    """Calculates the approximate memory usage of an object in bytes."""
    return sys.getsizeof(obj)

# Code Block : Set Initial State (Must handle dynamic inputs)
def set_initial_state(start_node):
    """
    Sets the initial state for the search.
    State is a tuple: (current_location, tuple_of_visited_locations)
    The tuple is used for hashing/set membership.
    """
    # Initial state: (Start node, (Start node,))
    return (start_node, tuple([start_node]))

# Code Block : Set the matrix for transition & cost (as relevant for the given problem)
# The graph structure defined above (GRAPH) serves as the transition and cost matrix.

# Pre-calculate minimum edge cost in the whole graph for the heuristic
# This is a key part of the admissible heuristic (h(n) >= 0)
ALL_EDGE_COSTS = []
for neighbors in GRAPH.values():
    ALL_EDGE_COSTS.extend(neighbors.values())
MIN_EDGE_COST = min(ALL_EDGE_COSTS) # This is 5

def get_transition_cost(loc1, loc2):
    """Returns the cost of transition between loc1 and loc2."""
    return GRAPH.get(loc1, {}).get(loc2, float('inf'))


# Code Block : Write function to design the Transition Model/Successor function. Ideally this would be called while search algorithms are implemented
def successor_function(current_state):
    """
    Generates successor states from the current state.
    Successor state: ((next_location, new_visited_set), cost_to_move)
    """
    current_location, visited_set_tuple = current_state
    
    # Visited set for quick lookup
    visited_set = set(visited_set_tuple)
    
    successors = []
    
    # Iterate through all neighbors
    for next_location, cost in GRAPH.get(current_location, {}).items():
        # Only move to unvisited locations, UNLESS all locations have been visited
        # and we are now looking for the final transition back to the start.
        
        # If all nodes visited, the only valid successor is the start node
        if len(visited_set) == NUM_LOCATIONS:
            if next_location == START_NODE:
                # Add the move back to start
                new_visited_set = visited_set | {next_location} # Mark start as visited again (optional, but consistent)
                successors.append(
                    ((next_location, tuple(sorted(list(new_visited_set)))), cost)
                )
            continue
        
        # Standard TSP step: move to an unvisited node
        if next_location not in visited_set:
            new_visited_set = visited_set | {next_location}
            successors.append(
                ((next_location, tuple(sorted(list(new_visited_set)))), cost)
            )

    return successors

# Code block : Write fucntion to handle goal test (Must handle dynamic inputs). Ideally this would be called while search algorithms are implemented
def goal_test(current_state):
    """
    Checks if the current state is the goal state.
    Goal state: All locations have been visited and the current location is the START_NODE.
    """
    current_location, visited_set_tuple = current_state
    
    # Check 1: Must be back at the start node
    if current_location != START_NODE:
        return False
        
    # Check 2: All locations must have been visited.
    # The initial state has (Start, (Start,)). A full tour has (Start, (All locations, Start)). 
    # The set of unique locations in the visited set must equal the total number of locations.
    if len(set(visited_set_tuple)) == NUM_LOCATIONS:
        return True
        
    return False

# Heuristic Function h(n) for RBFS
def h(state):
    """
    Admissible Heuristic for the TSP variant:
    h(n) = min_cost_from_current_to_unvisited + min_edge_cost * (number_of_unvisited_nodes - 1)
    """
    current_location, visited_set_tuple = state
    visited_set = set(visited_set_tuple)
    
    unvisited_locations = ALL_LOCATIONS - visited_set
    
    # If all nodes are visited, the remaining cost is the cost to return to start
    if not unvisited_locations:
        # Cost to return to start
        return get_transition_cost(current_location, START_NODE) if current_location != START_NODE else 0

    # 1. Minimum cost from current_location to any unvisited location
    min_cost_to_unvisited = float('inf')
    for unvisited in unvisited_locations:
        cost = get_transition_cost(current_location, unvisited)
        if cost < min_cost_to_unvisited:
            min_cost_to_unvisited = cost

    # 2. Minimum cost for the remaining (num_unvisited - 1) edges between unvisited nodes
    num_remaining_stops = len(unvisited_locations)
    
    # If there's only one unvisited node, the last leg is from that node back to the start.
    if num_remaining_stops == 1:
        last_unvisited = list(unvisited_locations)[0]
        min_cost_between_unvisited = get_transition_cost(last_unvisited, START_NODE)
    else:
        # A simple admissible lower bound: (num_remaining_stops - 1) edges * MIN_EDGE_COST
        min_cost_between_unvisited = MIN_EDGE_COST * (num_remaining_stops - 1)
        # We must also account for the edge back to the start from one of the unvisited nodes.
        min_cost_to_start_from_unvisited = float('inf')
        for unvisited in unvisited_locations:
             cost = get_transition_cost(unvisited, START_NODE)
             if cost < min_cost_to_start_from_unvisited:
                 min_cost_to_start_from_unvisited = cost
        
        # The total remaining cost is at least min_cost_to_unvisited + min_cost_between_unvisited + min_cost_to_start_from_unvisited
        # For simplicity and robust admissibility, we use a simpler sum:
        # min_cost_to_unvisited + min_edge_cost * (number_of_unvisited_nodes) 
        # (The number of edges in the remaining path is exactly the number of unvisited nodes, counting the return to start)
        return min_cost_to_unvisited + MIN_EDGE_COST * num_remaining_stops


    # The admissible estimate for the total remaining cost
    return min_cost_to_unvisited + min_cost_between_unvisited


# Utility Node class for search algorithms
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state  # (current_location, visited_set_tuple)
        self.parent = parent
        self.action = action  # (from_loc, to_loc)
        self.path_cost = path_cost  # g(n)
        self.f = self.path_cost + h(state) # f(n) = g(n) + h(n)

    def __lt__(self, other):
        # Comparison for priority queue (used conceptually in RBFS)
        return self.f < other.f
    
    def __repr__(self):
        return f"Node(Location: {self.state[0]}, Cost: {self.path_cost}, F: {self.f})"

# Utility to reconstruct the path
def reconstruct_path(node):
    """Reconstructs the path from the goal node back to the root."""
    path = []
    current_node = node
    while current_node:
        path.append(current_node.state[0])
        current_node = current_node.parent
    return path[::-1] # Reverse to get start-to-goal path


### 2.	Definition of Algorithm 1 (Depth First Search)

# Code Block : Function for algorithm 1 implementation
def depth_first_search_tsp(start_node):
    """
    Depth First Search to find the shortest complete tour (TSP variant).
    Uses a state of (location, visited_set_tuple) to handle the TSP requirement.
    Since DFS is uninformed, it finds all complete tours and returns the minimum cost one.
    """
    global DFS_NODES_EXPLORED, DFS_MAX_MEMORY
    
    DFS_NODES_EXPLORED = 0
    DFS_MAX_MEMORY = 0
    
    initial_state = set_initial_state(start_node)
    
    # DFS requires a stack for the fringe (LIFO)
    # Fringe stores (Node, Path_cost)
    fringe = deque([Node(initial_state)])
    
    # A set to track visited states (for simple cycle detection within the same visited set, not mandatory for full TSP)
    # The state (location, visited_set_tuple) already handles repeat locations with different visited sets.
    
    best_tour = None
    min_cost = float('inf')
    
    # Store all completed tours
    all_tours = []

    start_time = time.perf_counter()
    
    while fringe:
        current_node = fringe.pop()
        DFS_NODES_EXPLORED += 1
        
        # Update max memory usage (estimate)
        current_memory = get_memory_size(fringe) + get_memory_size(all_tours)
        if current_memory > DFS_MAX_MEMORY:
            DFS_MAX_MEMORY = current_memory
        
        if goal_test(current_node.state):
            # Goal is found: a complete tour ending at the start node
            path = reconstruct_path(current_node)
            cost = current_node.path_cost
            all_tours.append({"path": path, "cost": cost})
            # Continue search to find a better (minimum cost) tour
            continue

        # Check for path length to prevent infinite loops (only needed if goal test is not a good termination condition)
        # In TSP, the path length is bounded by the number of nodes. A node is defined by (location, visited_set), so cycles are avoided naturally.

        # Generate successors
        successors = successor_function(current_node.state)
        
        # Add unvisited successors to the fringe (LIFO order for DFS)
        for successor_state, cost_to_move in successors:
            
            # The new path cost is g(n) + cost_to_move
            new_path_cost = current_node.path_cost + cost_to_move
            
            # Create the child node
            child_node = Node(
                state=successor_state,
                parent=current_node,
                action=(current_node.state[0], successor_state[0]),
                path_cost=new_path_cost
            )
            
            fringe.append(child_node)

    end_time = time.perf_counter()
    total_time = end_time - start_time
    
    # Find the best tour from all found tours
    if all_tours:
        best_tour_data = min(all_tours, key=lambda x: x['cost'])
        return best_tour_data['path'], best_tour_data['cost'], DFS_NODES_EXPLORED, DFS_MAX_MEMORY, total_time
    else:
        return None, float('inf'), DFS_NODES_EXPLORED, DFS_MAX_MEMORY, total_time

### 3.	Definition of Algorithm 2 (Recursive Best First Search)

# Code Block : Function for algorithm 2 implementation
def recursive_best_first_search_tsp(start_node):
    """
    Recursive Best First Search (RBFS) implementation for the TSP variant.
    Guarantees optimality with the admissible heuristic h(n).
    """
    global RBFS_NODES_EXPLORED, RBFS_MAX_MEMORY
    
    RBFS_NODES_EXPLORED = 0
    RBFS_MAX_MEMORY = 0
    
    initial_state = set_initial_state(start_node)
    
    # Initial node: g(n)=0, f(n)=h(n)
    initial_node = Node(initial_state)

    start_time = time.perf_counter()
    
    # Start the recursive search with an initial F-limit of infinity
    result, f_limit_used = rbfs(initial_node, float('inf'))
    
    end_time = time.perf_counter()
    total_time = end_time - start_time
    
    if result is not None:
        path = reconstruct_path(result)
        cost = result.path_cost
        return path, cost, RBFS_NODES_EXPLORED, RBFS_MAX_MEMORY, total_time
    else:
        return None, float('inf'), RBFS_NODES_EXPLORED, RBFS_MAX_MEMORY, total_time

def rbfs(node, f_limit):
    """The recursive function for RBFS."""
    global RBFS_NODES_EXPLORED, RBFS_MAX_MEMORY
    RBFS_NODES_EXPLORED += 1
    
    # Update max memory usage (estimate: based on current node and f_limit)
    current_memory = get_memory_size(node) + sys.getsizeof(f_limit)
    if current_memory > RBFS_MAX_MEMORY:
        RBFS_MAX_MEMORY = current_memory
        
    if goal_test(node.state):
        return node, node.f  # Return the goal node and its f-cost

    # Get successors (g(s) + cost_to_move = new_path_cost)
    successors_data = successor_function(node.state)
    
    if not successors_data:
        # Node has no children, return failure with the node's f-cost
        return None, float('inf')

    # Create child nodes and store in a list for sorting
    children = []
    for successor_state, cost_to_move in successors_data:
        new_path_cost = node.path_cost + cost_to_move
        child_node = Node(
            state=successor_state,
            parent=node,
            action=(node.state[0], successor_state[0]),
            path_cost=new_path_cost
        )
        # Important: f-cost of child node must be max(f-cost of parent, g(child)+h(child)) 
        # (This is a common, though optional, feature of RBFS/IDA* to ensure consistency)
        child_node.f = max(child_node.f, node.f) 
        children.append(child_node)

    # Main RBFS loop
    while True:
        # Sort children by f-cost (best-first)
        children.sort(key=lambda x: x.f)
        
        best = children[0]  # The best successor
        if len(children) > 1:
            alternative = children[1].f # The second-best f-cost
        else:
            alternative = float('inf') # No alternative

        # Check if the best node's f-cost exceeds the current f-limit
        if best.f > f_limit:
            # All successors exceed the current limit, back-track
            return None, best.f

        # Recursive call: explore the best path, limited by the alternative cost
        result, new_f = rbfs(best, min(f_limit, alternative))

        # If a solution is found, return it
        if result is not None:
            return result, new_f
        
        # If the search failed, update the best node's f-cost and loop
        best.f = new_f
        # Loop continues to re-sort and try again

### DYNAMIC INPUT

# IMPORTANT : Dynamic Input must be got in this section. Display the possible states to choose from:
# This is applicable for all the relevent problems as mentioned in the question.

# Code Block : Function & call to get inputs (start/end state)
def get_user_input():
    """Displays possible locations and sets the start node."""
    print("Possible Tourist Locations (Nodes):")
    sorted_locations = sorted(list(ALL_LOCATIONS))
    for i, loc in enumerate(sorted_locations):
        print(f"  {i+1}. {loc}")
        
    # As per problem, the starting point is fixed as Indira Gandhi Airport
    start = START_NODE
    print(f"\nSetting Start Node as: {start} (as per problem statement)")
    print("Goal is to visit ALL locations and return to the Start Node.")
    return start

# Get the fixed start node
try:
    AGENT_START_NODE = get_user_input()
except Exception as e:
    print(f"Error during input setup: {e}")
    AGENT_START_NODE = START_NODE # Fallback

### 4.	Calling the search algorithms
# (For bidirectional search in below sections first part can be used as per Hint provided. Under second section other combinations as per Hint or your choice of 2 algorithms can be called .As an analyst suggest suitable approximation in the comparitive analysis section)

# Invoke algorithm 1 (Should Print the solution, path, cost etc., (As mentioned in the problem))
print("\n--- Algorithm 1: Depth First Search (DFS) for Optimal Tour ---")
dfs_path, dfs_cost, dfs_nodes, dfs_memory, dfs_time = depth_first_search_tsp(AGENT_START_NODE)

if dfs_path:
    print("Path Taken (Optimal Tour Found by DFS):")
    print(" -> ".join(dfs_path))
    print(f"Total Cost of the Tour: {dfs_cost}")
else:
    print("No complete tour found by DFS.")

# Invoke algorithm 2 (Should Print the solution, path, cost etc., (As mentioned in the problem))
print("\n--- Algorithm 2: Recursive Best First Search (RBFS) for Optimal Tour ---")
rbfs_path, rbfs_cost, rbfs_nodes, rbfs_memory, rbfs_time = recursive_best_first_search_tsp(AGENT_START_NODE)

if rbfs_path:
    print("Path Taken (Optimal Tour Found by RBFS):")
    print(" -> ".join(rbfs_path))
    print(f"Total Cost of the Tour: {rbfs_cost}")
else:
    print("No complete tour found by RBFS.")

### 5.	Comparitive Analysis (Time and Space Complexity)

# Code Block : Print the Time & Space complexity of algorithm 1
print("\n--- DFS Complexity Analysis ---")
print(f"Nodes Explored (Time Complexity proxy): {dfs_nodes} nodes")
print(f"Max Memory Usage (Space Complexity proxy): {dfs_memory} bytes")
print(f"Execution Time: {dfs_time:.6f} seconds")

# Theoretical Complexities:
# State space size S = N * 2^(N-1) for N nodes
# Time Complexity: O(b^m) where b is branching factor and m is max depth. 
# For TSP, max depth m=N and b<=N, so Time is O(N!). 
# Since we are checking ALL solutions for optimality, the complexity is high.
print("Theoretical Time Complexity: O(b^m) or O(N!) for all paths (TSP variant)")
print("Theoretical Space Complexity: O(b*m) or O(N) for the path taken")


# Code Block : Print the Time & Space complexity of algorithm 2
print("\n--- RBFS Complexity Analysis ---")
print(f"Nodes Explored (Time Complexity proxy): {rbfs_nodes} nodes")
print(f"Max Memory Usage (Space Complexity proxy): {rbfs_memory} bytes")
print(f"Execution Time: {rbfs_time:.6f} seconds")

# Theoretical Complexities:
# Time Complexity: O(b^(d/epsilon)) (Similar to A*, but uses less memory). For poor heuristics, it can be O(N!).
# Space Complexity: O(b*d) or O(N) (linear space in the depth of the optimal solution).
print("Theoretical Time Complexity: O(b^(d/epsilon)) (depends on heuristic quality), potentially O(N!) in worst case (TSP variant)")
print("Theoretical Space Complexity: O(b*d) or O(N) (linear space in the depth of the optimal solution)")


### 6.	Provide your comparitive analysis or findings in no more than 3 lines in below section

print("\n--- Comparative Findings ---")
print("DFS explores more nodes but is simpler, guaranteeing optimality only by checking all tours, which is computationally expensive.")
print("RBFS is **optimal** and **space-efficient**, only expanding nodes that are promising due to the admissible heuristic.")
print("The DFS implementation finds the true optimal path but by searching a much larger space, whereas RBFS finds it much faster by being informed.")

# Comparison : DFS is an uninformed search that explores all deep paths; RBFS is informed and guarantees optimality with the admissible heuristic.
#
# RBFS has a much better time complexity due to the informed search (less nodes explored) while maintaining O(N) space complexity.
#
# For small graphs like this TSP variant, both yield the same optimal path, but RBFS is theoretically and practically superior in efficiency.

Possible Tourist Locations (Nodes):
  1. Akshardham Temple
  2. Humayun's Tomb
  3. India Gate
  4. Indira Gandhi Airport
  5. Jama Masjid
  6. Lotus Temple
  7. Qutub Minar
  8. Red Fort

Setting Start Node as: Indira Gandhi Airport (as per problem statement)
Goal is to visit ALL locations and return to the Start Node.

--- Algorithm 1: Depth First Search (DFS) for Optimal Tour ---
Path Taken (Optimal Tour Found by DFS):
Indira Gandhi Airport -> Qutub Minar -> Lotus Temple -> Humayun's Tomb -> Akshardham Temple -> Jama Masjid -> Red Fort -> India Gate -> Indira Gandhi Airport
Total Cost of the Tour: 110

--- Algorithm 2: Recursive Best First Search (RBFS) for Optimal Tour ---
Path Taken (Optimal Tour Found by RBFS):
Indira Gandhi Airport -> Qutub Minar -> Lotus Temple -> Humayun's Tomb -> Akshardham Temple -> Jama Masjid -> Red Fort -> India Gate -> Indira Gandhi Airport
Total Cost of the Tour: 110

--- DFS Complexity Analysis ---
Nodes Explored (Time Complexity proxy): 164 nodes
Max 